# 02 — Campaign execution

Runs the placement-as-QACO campaign against the OpenBinding stack. Prerequisite:
the corpus generated by `01_dataset_preprocessing.ipynb`.

## Experimental design recap

| Variable | Domain | Size |
|---|---|---|
| Algorithm | `minizinc-csp` (exact, Gecode), `random-search` (**baseline**), `evolutionary-heuristics` (NSGA-II, MONO) | 3 |
| Objective | canonical weighted sum `J = 0.33·loss(latency) + 0.34·loss(cost) + 0.33·loss(security)` | 1 |
| Instances | 3 applications × 35 sizes (50–220) × seed 146588263 | 105 |
| Repetitions | exact ×1; stochastic ×10 (seeds 1–10) | 21 runs/instance |
| **Total** | | **2 205 runs** |

**GA configuration** — population size and variation operators of NSGA-II were selected in a
preliminary 3-way pilot on the hard application (`pilot_ga.py`, results in
`out/results/pilot_ga.csv`); the chosen values live in `campaign.campaign_specs()` and are
printed by the health-check cell below.

**Stopping criterion** — one wall-clock budget **T = 300 s shared by every algorithm**
(`campaign.TIME_BUDGET_MS`). Heuristics additionally guarantee **≥ 1 000 evaluations**
(the standard literature budget — the fallback measurement when the exact solver fails).

**Anytime protocol** — every solver returns the best solution it was considering when the
budget expires: the exact engine returns its last incumbent (status `SATISFIED` when optimality
is unproven; `UNSATISFIABLE`/`UNKNOWN` with zero solutions when it never found one), and the
heuristics return their best sample even when infeasible (flagged by the reference evaluator).
Best-so-far traces record `(eval_index, elapsed_ms)` per improvement, so any cutoff τ ≤ T is
recoverable offline.

Every returned binding is re-evaluated by the **gateway reference evaluator** (single source of
truth); `runs.csv` stores only canonical objective values.

## 0. Stack startup and health check

```bash
# from the repository root
docker compose --profile dev up -d --build gateway-dev engine-minizinc \
  engine-random-search engine-evolutionary-heuristics engine-many-heuristic
```

In [1]:
import sys
import time
from pathlib import Path

import httpx
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'docker-compose.yml').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'experimentation/icsoc'))

import campaign

engines = httpx.get(f'{campaign.DEFAULT_GATEWAY}/v1/engines', timeout=10).json()
for engine in engines:
    print(f"{engine['id']}: active={engine['active']}")
required = {'minizinc-csp', 'random-search', 'evolutionary-heuristics'}
assert all(e['active'] for e in engines if e['id'] in required), 'start the docker stack first'

n_instances = len(list(campaign.iter_instances()))
print(f'corpus: {n_instances} instances | shared budget T = {campaign.TIME_BUDGET_MS/1000:.0f}s | '
      f'{len(campaign.campaign_specs())} runs per instance')
for spec in campaign.campaign_specs():
    if spec.seed in (None, 1):
        print(f'{spec.engine}: {spec.options}')


minizinc-csp: active=True
random-search: active=True
many-heuristic: active=True
evolutionary-heuristics: active=True
corpus: 105 instances | shared budget T = 300s | 21 runs per instance
minizinc-csp: {'time_limit_ms': 300000, 'intermediate_solutions': True}
random-search: {'iterations_count': 1000, 'time_budget_ms': 300000, 'seed': 1}
evolutionary-heuristics: {'algorithm': 'NSGAII', 'operators': 'UNIFORM', 'population_size': 20, 'max_evaluations': 1000, 'time_budget_ms': 300000, 'seed': 1}


## 1. Pilot (optional): smallest vs largest instance

One pass over the two extreme instances to sanity-check the pipeline and extrapolate the campaign
duration before committing to the full run. Writes to a separate `results-pilot` directory.

In [ ]:
RUN_PILOT = False  # flip to True to run (~35 min: two exact runs may use the full budget)

if RUN_PILOT:
    pilot_dir = campaign.DEFAULT_RESULTS.parent / 'results-pilot'
    campaign.run_campaign(results_dir=pilot_dir, applications={'stockOrch'}, sizes={50})
    campaign.run_campaign(results_dir=pilot_dir, applications={'arOrch'}, sizes={220})
    pilot_df = pd.read_csv(pilot_dir / 'runs.csv')
    display(pilot_df.groupby(['application', 'engine'])[['wall_time_s', 'objective_value']]
            .agg(['mean', 'max']))
    T = campaign.TIME_BUDGET_MS / 1000
    print(f'Campaign upper bound: {105 * 21 * T / 3600:.0f} h sequential, '
          f'~{105 * 21 * T / 3600 / 3:.0f} h with 3 engine lanes')

## 2. Full campaign — launch commands

Heuristic runs consume the full budget T, so the recommended execution is **three parallel engine
lanes** from three terminals. Each lane is sequential inside its engine container, which keeps the
stochastic runs reproducible (one JVM-global RNG per engine):

```bash
# terminal 1 — exact lane (fastest: finishes early when optimality is proved)
.venv/bin/python experimentation/icsoc/campaign.py --engines minizinc-csp

# terminal 2 — baseline lane
.venv/bin/python experimentation/icsoc/campaign.py --engines random-search

# terminal 3 — evolutionary lane
.venv/bin/python experimentation/icsoc/campaign.py --engines evolutionary-heuristics
```

Results are **appended incrementally** to `out/results/runs.csv` and `out/results/traces.csv`;
re-running a lane resumes from the last recorded run. Useful variants:

```bash
# shorter horizon (traces still allow any cutoff <= T offline)
.venv/bin/python experimentation/icsoc/campaign.py --engines random-search --time-budget-ms 60000

# shard by application for a cluster
.venv/bin/python experimentation/icsoc/campaign.py --engines evolutionary-heuristics --applications arOrch
```

The cell below runs everything **sequentially in-notebook** instead (single lane; much slower).

In [ ]:
RUN_IN_NOTEBOOK = False  # prefer the three-lane CLI above
if RUN_IN_NOTEBOOK:
    campaign.run_campaign(verbose=True)

## 3. Progress monitoring

Re-run this cell at any time (equivalent CLI: `campaign.py --status`).

In [ ]:
campaign.print_status()

runs_csv = campaign.DEFAULT_RESULTS / 'runs.csv'
if runs_csv.exists():
    runs = pd.read_csv(runs_csv)
    done = runs[runs.status == 'ok'].groupby(['engine', 'application']).size().unstack(fill_value=0)
    ax = done.T.plot(kind='bar', figsize=(7, 3), rot=0,
                     title='completed runs per engine and application')
    ax.set_ylabel('runs')

## 4. Notes

- **Resumability**: run ids are deterministic (`instance|engine#seed`); interrupted lanes restart
  where they stopped. To redo a run, delete its row from `runs.csv`.
- **Integrity**: `oracle_match=False` rows indicate an engine reported an objective that the
  reference evaluator does not reproduce — investigate before trusting results (expected: none).
- When all 2 205 runs are recorded, continue with `03_results_evaluation.ipynb`.